In [2]:

from mmdet.registry import VISUALIZERS
import sys
from pathlib import Path
import torch 

sys.path.append('/Data_large/marine/PythonProjects/MMDET/notebooks/Tools')
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs/custom_components')
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs')


from mmdet.apis import init_detector, inference_detector
from mmengine.config import Config
from mmengine.runner import load_checkpoint
from mmcv.transforms import Compose

from cam import EigenCAM
import rasterio as rio 
import numpy as np

def estimate_vmin_vmax(cam, percentiles=[5, 95]):
    """
    Estimates vmin and vmax values for the heatmap.

    Parameters:
    - cam (numpy.ndarray): The heatmap.

    Returns:
    - tuple: A tuple containing the vmin and vmax values.
    """
    vmin = np.percentile(cam, percentiles[0])
    vmax = np.percentile(cam, percentiles[1])
    return vmin, vmax

# Required by loader
def read_tif(file_path, band_indices):
    """
    Reads specified bands from a TIFF file.

    Parameters:
    - file_path (str): Path to the .tif file.
    - band_indices (list of int): Indices of the bands to read.

    Returns:
    - numpy.ndarray: A numpy array containing the stacked band data.
    """
    data = []
    with rio.open(file_path) as src:
        for index in band_indices:
            data.append(src.read(index))
    stacked_data = np.stack(data, axis=0)
    return np.transpose(stacked_data, (1, 2, 0))

DATA_PATH_VEN = '/Data_large/marine/Datasets/VENuS/ds_L0/perfect'
DATA_PATH_SEN = '/Data_large/marine/Datasets/VDS2Raw/imgs'

TIFF_VEN = list(Path(DATA_PATH_VEN).rglob('*.tif'))
TIFF_SEN = list(Path(DATA_PATH_SEN).rglob('*.tif'))


# Specify the path to model config and checkpoint file
config_file = '/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b10/18_BS_3_LR_0.0008_ME_30_OPT_SGD/vfnet_r18.py'
checkpoint_file = '/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b10/18_BS_3_LR_0.0008_ME_30_OPT_SGD/epoch_30.pth'

# build the model from a config file and a checkpoint file
DetModel = init_detector(config_file, checkpoint_file, device='cuda:0')

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

cfg = Config.fromfile(config_file)
cfg = cfg.copy()
test_pipeline = cfg.test_dataloader.dataset.pipeline

model = DetModel
checkpoint = checkpoint_file
checkpoint = load_checkpoint(model, checkpoint, map_location='cpu')


checkpoint_meta = checkpoint.get('meta', {})
dataset_meta = checkpoint_meta['dataset_meta']['classes']
model.dataset_meta = dataset_meta

model.to(device)
model.eval()

Idx = 157
tiffSel = None
test_pipeline = Compose(test_pipeline)
if tiffSel is None:
    data_ = dict(img_path=TIFF_VEN[Idx], img_id=0)
    ORIGINAL_IMG = read_tif(file_path=TIFF_VEN[Idx], band_indices=[5])

else:
    data_ = dict(img_path=tiffSel, img_id=0)
    ORIGINAL_IMG = read_tif(file_path=tiffSel, band_indices=[5])
    

data_ = test_pipeline(data_)
data_['inputs'] = [data_['inputs']]
data_['data_samples'] = [data_['data_samples']]


# forward the model
with torch.no_grad():
    results = model.test_step(data_)[0]


Loads checkpoint by local backend from path: /Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b10/18_BS_3_LR_0.0008_ME_30_OPT_SGD/epoch_30.pth
Loads checkpoint by local backend from path: /Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Single/perfect_b10/18_BS_3_LR_0.0008_ME_30_OPT_SGD/epoch_30.pth


/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [3]:
x = data_['inputs']
inp = x[0].to(device).unsqueeze(0)
inp.shape

torch.Size([1, 1, 2048, 2048])

Forward pass and export to onnx

In [16]:
# Forward pass
with torch.no_grad():
    outputs = model(inp)


# Export the model to ONNX format
torch.onnx.export(model.backbone, inp, 'model_encoder.onnx', opset_version=11)
inp_neck = list(model.backbone(inp))
torch.onnx.export(model.neck, inp_neck, 'model_neck.onnx', opset_version=11)

============= Diagnostic Run torch.onnx.export version 2.0.0+cu118 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================



In [51]:
out_neck = model.neck(inp_neck)

torch.onnx.export(model.bbox_head, list(out_neck), 'model_heads.onnx', opset_version=11)





/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmcv/ops/deform_conv.py:335: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  input_pad = (x.size(2) < self.kernel_size[0]) or (x.size(3) <
/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmcv/ops/deform_conv.py:337: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if input_pad:
/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmcv/ops/deform_conv.py:218: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't re

============= Diagnostic Run torch.onnx.export version 2.0.0+cu118 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 10 WARNING 0 ERROR =======================
10 WARNING were not printed due to the log level.



In [5]:
for o in outputs[0]:
    print(o.shape)

torch.Size([1, 1, 256, 256])
torch.Size([1, 1, 128, 128])
torch.Size([1, 1, 64, 64])
torch.Size([1, 1, 32, 32])
torch.Size([1, 1, 16, 16])


In [6]:
results

<DetDataSample(

    META INFORMATION
    scale_factor: (1.0767613038906414, 0.8554720133667502)
    batch_input_shape: (2048, 2048)
    pad_shape: (2048, 2048)
    img_shape: (2048, 2048)
    gt_bboxes: HorizontalBoxes(
        tensor([], size=(0, 4)))
    homography_matrix: array([[1.0767612, 0.       , 0.       ],
               [0.       , 0.855472 , 0.       ],
               [0.       , 0.       , 1.       ]], dtype=float32)
    keep_ratio: False
    img_path: PosixPath('/Data_large/marine/Datasets/VENuS/ds_L0/perfect/ASH_L0_07558_20190105_MultiLayer_mask_OK.tif')
    gt_ignore_flags: array([], dtype=bool)
    gt_bboxes_labels: array([], dtype=int64)
    ori_shape: (2394, 1902)
    img: array([[  0.     ,   0.     ,   0.     , ...,   0.     ,   0.     ,
                  0.     ],
               [  0.     ,   0.     ,   0.     , ...,   0.     ,   0.     ,
                  0.     ],
               [  0.     ,   0.     ,   0.     , ...,   0.     ,   0.     ,
                  0.  